# 임베딩 실습 4일차: LLM 라벨링 (논문 데이터)

> 논문 제목과 초록에 LLM으로 **속성(연구 대상)**과 **극성(결과 방향)**을 붙이고, 검증과 비율까지 해 봅니다.

## 전체 흐름

```
코드북 (속성 정의) + JSON 스키마 (출력 모양)
  ↓ ① 연구 유형 고르기 (효과 · 측정 · 개관 · 전략)
  ↓ ② 유형을 참고해 속성과 극성, 근거 텍스트
  ↓ 검증: 파싱 → 스키마 → 재시도 1회 → 근거가 원문에 있는가
  ↓ 사람 판단과 비교 (재현율, 기준선)
  ↓ 대량 라벨 → 속성별 부정 비율 + 95% 구간 (순위는 구간이 겹치지 않을 때만)
```

## 고친 과정 한눈에 보기

| 단계 | 바꾼 것 | 결과 |
|---|---|---|
| 코드북 v1 | 리뷰 속성 → 논문 속성, 극성은 결과 방향 | 시작점 |
| 코드북 v2 | 어업·어선 추가, 기타 정의 명확화 | 심해 어류가 기타로 감 |
| 스키마 버그 수정 | 검사 함수가 옛날 스키마를 기억하던 문제 | 어업·어선이 정상 통과 |
| 프롬프트 v1 | 긴 근거 문장 복사 | 보류 4편, 근거 확인 0.56 |
| 프롬프트 v2 | 근거를 20~40자로 짧게 | 보류 5편, 0.38로 악화 (한국어 요약이 섞임) |
| 프롬프트 v3 | 라벨은 한국어, 근거는 영어 원문 그대로 | 근거 확인 0.90 |
| 프롬프트 v4 | 코드북 v2 반영 | 기타 정확, 가오리 논문은 라벨 누락 |
| 초록 품질 필터 | 150자 미만 1,106편 제외 | 분석 대상 28,311편 |
| 프롬프트 v5 | 어업 예시, 전략·개관은 중립 규칙 | 일부 개선, 다른 곳 회귀 |
| 두 단계 방식 | 연구 유형을 먼저 고르고 극성 결정 | 개관·전략 중립 개선, 효과 논문 오분류 |
| 유형 정의 구체화 | 효과 = 결과 확인, 측정 중 피해도 효과 | 10편 중 7편 정답, 근거 확인 1.00 |

**배운 것**
- 근거가 원문에 있다고 라벨이 맞는 건 아니에요. 둘은 따로 확인해요.
- 프롬프트에 규칙을 더하면 다른 곳이 깨질 수 있어요. 고칠 때마다 전체를 다시 확인해요.
- "짧게 쓰라"는 규칙이 번역을 부를 수 있어요.
- 자료 품질(깨진 초록)이 모델 성능보다 결과를 더 크게 바꿀 수 있어요.

## 1. 준비

- 데이터(works.csv)와 모델(Qwen3-4B-Instruct-2507)이 필요해요.
- 런타임은 **T4 GPU**로 설정하세요.

In [ ]:
!pip -q install transformers accelerate
import json
import re
import numpy as np
import pandas as pd
import torch
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

df = pd.read_csv('works.csv').dropna(subset=['title', 'abstract']).reset_index(drop=True)
df_ok = df[df['abstract'].str.len() >= 150].reset_index(drop=True)   # 초록 150자 미만 제외
print('전체:', len(df), '| 분석 대상:', len(df_ok))

name = 'Qwen/Qwen3-4B-Instruct-2507'
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map='cuda')
print('모델 준비 완료')

## 2. 코드북 v2

- 속성 7개: 상어·가오리, 연체류, 양식·식품, 보호·관리, 해양 환경, 어업·어선, 기타
- 극성 3개: 긍정(효과 확인), 부정(피해·악화 확인), 중립(그 밖)

**v2에서 바꾼 것**: 어업·어선을 추가했어요. 기타에는 상어·가오리가 아닌 어류와 일반 해양 생물을 명시했어요.

In [ ]:
ASPECTS = {
    '상어·가오리': {'정의': '상어나 가오리류가 주된 대상인 연구',
                  '포함': '상어·가오리 종, 개체군, 어획, 보전',
                  '제외': '상어가 아닌 어종이 주제인 연구'},
    '연체류': {'정의': '문어·오징어 등 두족류가 주된 대상인 연구',
              '포함': '두족류의 생물학, 산업, 어획',
              '제외': '두족류를 부수적으로만 언급한 연구'},
    '양식·식품': {'정의': '양식이나 수산 식품, 식량이 주된 대상인 연구',
                 '포함': '양식 생산, 양식 경제, 수산 식품',
                 '제외': '자연 어획만 다룬 연구'},
    '보호·관리': {'정의': '보호구역, 해양 정책, 공간 계획이 주된 대상인 연구',
                 '포함': '보호구역, 관리 제도, 계획',
                 '제외': '생물 자체의 생태만 다룬 연구'},
    '해양 환경': {'정의': '오염, 수질, 생태계 변화가 주된 대상인 연구',
                 '포함': '오염, 수질, 환경 영향',
                 '제외': '생물 분류가 주제인 연구'},
    '어업·어선': {'정의': '어선, 어구, 어업 경제·운영이 주된 대상인 연구',
                 '포함': '어선 경제성, 어구, 어업 운영·규제, 어획 노력',
                 '제외': '양식 생산이나 수산 식품이 주제인 연구'},
    '기타': {'정의': '위 속성 어디에도 속하지 않는 연구',
            '포함': '상어·가오리가 아닌 어류(예: 심해 어류)와 일반 해양 생물 연구, 교육 자료',
            '제외': '위 속성 중 하나에 해당하면 기타가 아님'},
}

POLARITY = {
    '긍정': '정책·관리·기술의 효과가 확인됨',
    '부정': '자원 감소, 오염, 피해, 위협이 확인됨',
    '중립': '위 둘 중 어느 쪽도 아님 (측정, 분류, 방법 연구)',
}
print('속성', len(ASPECTS), '개 · 극성', len(POLARITY), '개')

## 3. JSON 스키마와 검사 함수

**고친 점**: 처음에는 검사 함수가 옛날 스키마(속성 6개)를 기본값으로 기억하고 있어서, 어업·어선을 맞게 골라도 틀렸다고 판정했어요. 호출할 때마다 최신 스키마를 쓰도록 바꿨어요.

In [ ]:
SCHEMA = {
    'type': 'object',
    'required': ['aspects'],
    'properties': {
        'aspects': {
            'type': 'array',
            'items': {
                'type': 'object',
                'required': ['aspect', 'polarity', 'evidence'],
                'properties': {
                    'aspect': {'enum': list(ASPECTS.keys())},
                    'polarity': {'enum': list(POLARITY.keys())},
                    'evidence': {'type': 'string'},
                },
            },
        },
    },
}


def schema_errors(obj, schema=None):
    schema = schema or SCHEMA
    errs = []

    def walk(x, sch, path):
        if 'enum' in sch:
            if x not in sch['enum']:
                errs.append(f'{path} 값 {x!r} — 허용 목록 밖')
            return
        if sch.get('type') == 'object':
            for k in sch.get('required', []):
                if k not in x:
                    errs.append(f'{path}.{k} 없음')
            for k, sub in sch.get('properties', {}).items():
                if k in x:
                    walk(x[k], sub, f'{path}.{k}')
        elif sch.get('type') == 'array':
            for i, item in enumerate(x):
                walk(item, sch['items'], f'{path}[{i}]')
        elif sch.get('type') == 'string' and not isinstance(x, str):
            errs.append(f'{path} 는 문자열이어야 함')

    walk(obj, schema, '$')
    return errs


print(schema_errors({'aspects': [{'aspect': '디자인', 'polarity': '부정', 'evidence': 'x'}]}))

## 4. 두 단계 프롬프트 (최종 버전)

**① 연구 유형 고르기**: 효과, 측정, 개관, 전략 중 하나
**② 속성과 극성**: 유형을 참고해 정하기

**고친 점**
- 라벨은 한국어, 근거는 **영어 원문을 그대로 복사**해요.
- 효과는 실제 수치 변화나 결과가 확인된 논문이에요. 측정 중 피해가 확인되면 효과로 봐요.
- 극성은 효과 논문만 긍정·부정, 나머지는 중립이에요.

In [ ]:
def squeeze(s):
    return re.sub(r'\s+', ' ', s).strip()


def ask(system, user):
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)   # 온도 0
    raw = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    m = re.search(r'\{.*\}', raw, re.S)
    try:
        return json.loads(m.group(0)), None
    except Exception as e:
        return None, f'JSON 파싱 실패 ({e})'


TYPES = {
    '효과': '실제 수치 변화나 결과가 확인된 논문. 측정 중 피해나 악화가 확인된 경우도 여기에 넣음',
    '측정': '결과 변화 없이 수치만 재거나 자원·생태를 추정한 논문',
    '개관': '여러 연구를 정리하거나 전체 현황을 소개한 논문',
    '전략': '결과 수치 없이 방법이나 방향만 제안한 논문',
}
type_text = '\n'.join(f'- {k}: {v}' for k, v in TYPES.items())
codebook_text = '\n'.join(
    f"- {a}: {v['정의']} / 포함: {v['포함']} / 제외: {v['제외']}" for a, v in ASPECTS.items())
polar_rule = ('연구 유형이 효과인 경우에만 긍정 또는 부정을 쓰세요.\n'
              '연구 유형이 개관, 측정, 전략이면 polarity 는 반드시 중립입니다.')

SYS_TYPE = ('당신은 수산·해양 논문의 연구 유형을 고르는 보조입니다.\n유형 목록:\n' + type_text +
            '\n\n위 네 가지 중 하나만 골라 JSON 한 개로 답하세요.\n형식: {"type": "효과|측정|개관|전략"}')
SYS_LABEL = ('당신은 수산·해양 논문의 연구 대상과 결과 방향을 분류하는 보조입니다.\n속성 목록:\n' +
             codebook_text + '\n\n극성 규칙:\n' + polar_rule +
             '\n- aspect 는 위 속성 목록의 한국어 이름만 쓰세요.'
             '\n- evidence 는 초록(영어)에서 글자 그대로 복사한 짧은 구절(20~40자)만 쓰세요.'
             '\n- evidence 를 번역하거나 요약하지 마세요.'
             '\n- JSON 한 개만 답하세요.'
             '\n형식: {"aspects": [{"aspect": "...", "polarity": "...", "evidence": "..."}]}')


def label_one(title, abstract):
    """연구 유형 → 라벨 → 검증 → 재시도 1회. 실패하면 None (분모에는 남긴다)."""
    user = f'제목: {title}\n초록: {abstract[:600]}'
    t_obj, _ = ask(SYS_TYPE, user)
    t = t_obj.get('type') if t_obj else '측정'
    user2 = f'연구 유형: {t}\n' + user

    obj, err = ask(SYS_LABEL, user2)
    if obj is not None and not schema_errors(obj):
        return t, obj['aspects'], 'first'
    msg = err or '; '.join(schema_errors(obj))
    obj, err = ask(SYS_LABEL, user2 + f'\n\n이전 답의 오류: {msg}\n규칙에 맞는 JSON 한 개만 다시 답하세요.')
    if obj is not None and not schema_errors(obj):
        return t, obj['aspects'], 'retry'
    return t, None, 'fail'


print('프롬프트 준비 완료')

## 5. 10편으로 확인 (근거 확인)

**근거 확인**: 근거 텍스트가 제목이나 초록에 글자 그대로 있는지 봐요. 있으면 확인, 없으면 보류예요.

In [ ]:
sample10 = df_ok.sample(10, random_state=0)
rows10 = []
for i, row in sample10.iterrows():
    t, aspects, status = label_one(row['title'], row['abstract'])
    rows10.append((i, aspects))
    print(f'\n[{i}] {row["title"][:60]} · 유형 {t} · {status}')
    if aspects is None:
        continue
    src = squeeze(row['title']) + ' ' + squeeze(row['abstract'])
    for a in aspects:
        ev = squeeze(a['evidence'])
        print(f"  ({a['aspect']}, {a['polarity']}) {'근거 확인' if ev in src else '보류'} · 「{ev[:60]}」")

## 6. 사람 판단과 비교

사람이 정한 정답 10편으로 재현율을 계산해요. 기준선은 "늘 양식·식품·중립"만 붙였을 때의 값이에요.

**결과**: 재현율 70%(10개 중 7개), 기준선 20%(2개)

**주의**: 10편은 표본이 작아서 이 값은 참고용이에요.

In [ ]:
human = {
    9917: [('연체류', '중립')], 14408: [('어업·어선', '중립')], 1909: [('어업·어선', '중립')],
    10316: [('연체류', '부정')], 10203: [('연체류', '중립')], 16618: [('양식·식품', '중립')],
    759: [('어업·어선', '중립')], 22905: [('양식·식품', '긍정')], 21029: [('양식·식품', '중립')],
    19437: [('양식·식품', '긍정')],
}
llm = {i: {(a['aspect'], a['polarity']) for a in asp} for i, asp in rows10 if asp is not None}

hit, total = 0, 0
for i, truth in human.items():
    for pair in truth:
        total += 1
        hit += pair in llm.get(i, set())
print(f'사람 정답 {total}개 중 맞힌 수: {hit} → 재현율 {hit / total:.0%}')

baseline = sum(('양식·식품', '중립') in truth for truth in human.values())
print(f'기준선(늘 양식·식품·중립): {baseline}/{len(human)} → {baseline / len(human):.0%}')

## 7. 200편 대량 라벨과 95% 구간

**규칙**
- 분모는 보낸 200편이에요. 실패한 논문도 분모에 남겨요.
- 구간이 겹치지 않는 속성만 순위를 말해요.
- 0%인 구간은 정규 근사가 잘 맞지 않아 실제로는 더 넓을 수 있어요.

**결과 (T4 GPU, 약 10~20분)**: 첫 시도 통과 199편, 재시도 0편, 라벨 실패 1편

| 속성 | 부정 논문 수 | 비율 | 95% 구간 |
|---|---:|---:|---|
| 상어·가오리 | 7 | 3.5% | 1.0% ~ 6.0% |
| 해양 환경 | 3 | 1.5% | 0.0% ~ 3.2% |
| 연체류 | 2 | 1.0% | 0.0% ~ 2.4% |
| 양식·식품 | 1 | 0.5% | 0.0% ~ 1.5% |
| 보호·관리, 기타, 어업·어선 | 0 | 0.0% | 0.0% ~ 0.0% |

상어·가오리와 해양 환경의 구간이 겹쳐서 순위를 말할 수 없어요.

In [ ]:
sample200 = df_ok.sample(200, random_state=42)
N = len(sample200)
first, retry, fail = 0, 0, 0
recs = []
for i, row in sample200.iterrows():
    t, aspects, status = label_one(row['title'], row['abstract'])
    if status == 'first':
        first += 1
    elif status == 'retry':
        retry += 1
    else:
        fail += 1
    recs.append((i, aspects))

print(f'분모 {N}편 · 첫 시도 {first} · 재시도 {retry} · 라벨 실패 {fail}')

has_neg = defaultdict(set)
for i, aspects in recs:
    if aspects is None:
        continue
    for a in aspects:
        if a['polarity'] == '부정':
            has_neg[a['aspect']].add(i)

print('\n속성 · 부정 논문 수 · 비율 · 95% 구간')
for name in ASPECTS:
    k = len(has_neg[name])
    p = k / N
    margin = 1.96 * np.sqrt(p * (1 - p) / N)
    print(f'{name:<8} {k:>3}편 · {p:.1%} · {max(p - margin, 0):.1%} ~ {min(p + margin, 1):.1%}')

pd.DataFrame(
    [(i, a['aspect'], a['polarity'], a['evidence']) for i, asp in recs if asp for a in asp],
    columns=['논문', '속성', '극성', '근거'],
).to_csv('labels_200.csv', index=False, encoding='utf-8-sig')
print('labels_200.csv 저장 완료')

## 8. 비용 계산 (API로 보낸다면)

API 호출은 하지 않고, 토큰 수로 비용만 추정해요. 단가는 강의 자료의 gpt-5-nano 표준 단가(100만 토큰당 입력 0.05달러, 출력 0.40달러, 조회 2026-09-25)예요.

**결과**: 논문 한 편 입력 평균 1,842토큰 → 28,311편 표준 약 3.51달러, 배치 약 1.76달러

**주의**: 토큰 수는 Qwen 토크나이저 기준이라 OpenAI 토크나이저와 정확히 같지 않아요. 대략의 크기를 보는 용도예요. 비용이 싸더라도 보낼 곳의 데이터 정책은 따로 확인해야 해요.

In [ ]:
def ntok(s):
    return len(tok(s)['input_ids'])


sample_t = df_ok.sample(200, random_state=1)
sys_tok = ntok(SYS_TYPE) + ntok(SYS_LABEL)
user_tok = np.array([ntok(f"제목: {t}\n초록: {a[:600]}")
                     for t, a in zip(sample_t['title'], sample_t['abstract'])])

OUT_TOK = 80
N_TARGET = len(df_ok)
P_IN, P_OUT = 0.05, 0.40
in_per_paper = 2 * sys_tok + 2 * user_tok.mean()
cost_std = N_TARGET * (in_per_paper * P_IN + OUT_TOK * P_OUT) / 1e6
print(f'논문 한 편 입력 토큰(평균): {in_per_paper:.0f} · 출력 토큰(가정): {OUT_TOK}')
print(f'분석 대상 {N_TARGET:,}편 표준 비용: {cost_std:.2f}달러 · 배치 {cost_std / 2:.2f}달러')

## 9. 정리

**4일차에 한 일**
- 코드북을 리뷰용에서 논문용으로 바꾸고, 어업·어선을 추가했어요.
- 스키마 검사 버그를 고쳤어요.
- 프롬프트를 여러 번 고쳤어요. 최종은 두 단계(연구 유형 → 라벨)예요.
- 10편으로 근거 확인과 재현율을 봤어요. 재현율 70%, 기준선 20%였어요.
- 200편으로 속성별 부정 비율과 95% 구간을 구했어요. 순위를 말할 수 있는 속성은 없었어요.
- 비용을 토큰으로 계산했어요.

**남은 과제**
- 2번(게 채집)과 7번(저수지 자원 추정)의 속성이 여전히 틀려요. 경계 기준을 더 분명히 쓰는 게 다음 과제예요.
- 재시도와 라벨 실패 경로는 200편에서 거의 일어나지 않았어요. 더 어려운 자료에서 확인해 볼 필요가 있어요.
- 정확도(잘못 붙인 라벨 비율)는 사람이 더 많이 판단해야 알 수 있어요.

**팀 보고용 문장**
> 분석 대상 28,311편 가운데 200편을 무작위로 골라 LLM으로 라벨링했습니다. 대부분은 측정·개관·전략 연구라 중립으로 분류되었고, 부정 라벨은 상어·가오리 7편(3.5%), 해양 환경 3편(1.5%)이었습니다. 95% 구간이 겹쳐 속성 사이의 순위는 말할 수 없습니다.